Key Changes

    Annotated Folder: Added a new folder seamounts_annotated to store images with bounding boxes.
    Pixel Coordinates: Converted bounding box latitude/longitude to pixel coordinates relative to the image dimensions.
    Bounding Box Drawing: Used cv2.rectangle to overlay the bounding box on the image.

Output

    Original Images: Saved in the seamounts_galore folder.
    Annotated Images: Saved in the seamounts_annotated folder with green bounding boxes for visual inspection.
    Bounding Box CSV: Contains the bounding box coordinates for each image.

In [12]:
import os
import requests
import pandas as pd
import math
import cv2
import numpy as np
import time
import logging

# Parameters
csv_file = "seamounts.csv"  # Path to the .csv file
output_folder = "seamounts_galore"  # Folder to save downloaded images
annotated_folder = "seamounts_annotated"  # Folder to save annotated images
bbox_folder = "seamounts_bboxes"  # New folder to save bounding box images
bbox_file = "seamounts_bboxes.csv"  # File to save bounding box data
tile_pixels = 800  # Width and height of the image in pixels
api_base_url = "https://www.gmrt.org/services/ImageServer"
max_retries = 3  # Max retries for failed downloads

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Ensure output folders exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(annotated_folder, exist_ok=True)
os.makedirs(bbox_folder, exist_ok=True)

# Load CSV file and limit to 1000 random rows
try:
    df = pd.read_csv(csv_file)
    df = df.sample(n=1000, random_state=42).reset_index(drop=True)
    logging.info(f"Loaded {len(df)} records from {csv_file}.")
except Exception as e:
    logging.error(f"Error loading .csv file: {e}")
    exit()

# Prepare bounding box file
bbox_data = []

# Process each record
for index, row in df.iterrows():
    retries = 0
    while retries < max_retries:
        try:
            # Extract data from the row
            file_name = str(row["PEAKID"])  # Use PEAKID as the file name
            center_lon = float(row["LONG"])  # Longitude from LONG
            center_lat = float(row["LAT"])  # Latitude from LAT
            area_km2 = float(row["AREA2D"])  # Area in km^2

            # Calculate the radius in degrees from the area
            radius_deg = math.sqrt(area_km2 / math.pi) / 111.0  # 1 degree ≈ 111 km

            # Calculate bounding box coordinates
            x_min = center_lon - radius_deg
            x_max = center_lon + radius_deg
            y_min = center_lat - radius_deg
            y_max = center_lat + radius_deg

            # Debugging: Print calculated values
            logging.debug(f"PEAKID: {file_name}")
            logging.debug(f"Center Lat: {center_lat}, Center Lon: {center_lon}")
            logging.debug(f"Radius (degrees): {radius_deg}")
            logging.debug(f"Bounding Box: {x_min}, {y_min}, {x_max}, {y_max}")

            # Prepare API parameters for image download (using bounding box coordinates)
            params = {
                "minlatitude": y_min,
                "maxlatitude": y_max,
                "minlongitude": x_min,
                "maxlongitude": x_max,
                "width": tile_pixels,
                "mask": "false",
                "download": "true",
            }

            # Construct file paths
            bbox_file_path = os.path.join(bbox_folder, f"{file_name}_1.png")  # Save bbox-specific image

            # Download the image for the bounding box
            logging.info(f"Downloading bounding box image for PEAKID {file_name}...")
            response = requests.get(api_base_url, params=params)

            if response.status_code == 200:
                # Save the bounding box image
                with open(bbox_file_path, "wb") as file:
                    file.write(response.content)
                logging.info(f"Bounding box image saved: {bbox_file_path}")

                # Append bounding box data
                bbox_data.append([file_name, x_min, y_min, x_max, y_max])
            else:
                logging.error(f"Failed to download bounding box for PEAKID {file_name} - HTTP {response.status_code}")
                retries += 1
                time.sleep(2)  # Wait before retrying
                continue

            break  # Break the retry loop if successful

        except Exception as e:
            logging.error(f"Error processing row {index}: {e}")
            retries += 1
            time.sleep(2)  # Wait before retrying
            continue

# Save bounding box data to CSV
bbox_df = pd.DataFrame(bbox_data, columns=["PEAKID", "x_min", "y_min", "x_max", "y_max"])
bbox_df.to_csv(bbox_file, index=False)
logging.info(f"Bounding box data saved to {bbox_file}")

logging.info("Process complete.")


2024-12-21 15:15:54,973 - INFO - Loaded 1000 records from seamounts.csv.
2024-12-21 15:15:54,975 - INFO - Downloading bounding box image for PEAKID 1141217.0...
2024-12-21 15:15:58,044 - INFO - Bounding box image saved: seamounts_bboxes\1141217.0_1.png
2024-12-21 15:15:58,045 - INFO - Downloading bounding box image for PEAKID 3190447.0...
2024-12-21 15:16:00,756 - INFO - Bounding box image saved: seamounts_bboxes\3190447.0_1.png
2024-12-21 15:16:00,757 - INFO - Downloading bounding box image for PEAKID 830693.0...
2024-12-21 15:16:03,917 - INFO - Bounding box image saved: seamounts_bboxes\830693.0_1.png
2024-12-21 15:16:03,918 - INFO - Downloading bounding box image for PEAKID 3773774.0...
2024-12-21 15:16:05,385 - INFO - Bounding box image saved: seamounts_bboxes\3773774.0_1.png
2024-12-21 15:16:05,386 - INFO - Downloading bounding box image for PEAKID 2492990.0...
2024-12-21 15:16:06,920 - INFO - Bounding box image saved: seamounts_bboxes\2492990.0_1.png
2024-12-21 15:16:06,921 - INF